# TeCo: reader identity, fixation duration, and familiar-identity sentiment

**CPU → Run all. Choose Zyion for both Drive prompts.** No GPU or new extraction. Allow several hours; actual runtime is not benchmarked. Earlier sweep took about13minutes, but these new targets use a larger pool and more cases. Completed model fits resume.

## Targets and data
1. **Reader identity:** classify the12known readers. Chance balanced accuracy≈8.3%. Use all available accepted TeCo task1 cached epochs, regardless of sentiment label. Compare familiar identities with **unseen sentences**; unseen readers cannot be test classes in closed-set identity classification.
2. **Fixation duration:** short/long based on a fixed **200ms cutoff**; repeat with150and250ms as sensitivity analyses. Short is strictly below cutoff, long includes equality. Exclude missing/nonfinite/nonpositive duration and report counts. Primary200ms target has familiar, unseen-sentence and unseen-reader evaluations. Do not input duration or next-fixation delay, or residualized features containing them. Late EEG may classify a fixation that has already ended; this is not a prospective duration forecast.
3. **Original sentiment experiments with familiar identities:** positive/negative; positive/negative/neutral; neutral/nonneutral. Recover each from checksum-verified **original saved prediction rosters**, not current annotation files. Neutral means the original pooled-neutral selection, not neutral sentences only. Missing original roster is explicitly skipped. This does not change existing labels or results.

No word-length classification is included. The non-EEG comparator can still use word length, relative word position, sentence length and control-sentence flag; these are explanatory controls, not new target tasks.

## “Nothing unseen” means familiar identities, separate EEG events
In the **familiar** setting, every retained test **reader**, **word identity**, and **sentence identity** is present in training after purging. It does not require the same reader–word–sentence combination to occur twice. Different occurrences/readers may supply the familiar identities. Test events themselves never enter training. We remove training epochs within1000ms of held-out epochs in the same reader/sentence. If this or familiarity screening removes too much data, report an unavailable condition; do not relax the rule. The exact words, IDs and timing used for these checks are not classifier inputs.

All three identities are familiar **at the individual-event level before averaging**. Known reader/target labels form averages within partition. This resembles pseudotrial analysis; class-average sentiment/duration scores are not single-word deployment accuracy. Each inner validation split applies the same familiarity/purge rules.

This addresses the absence of identity holdouts, **not an exact replication of the paper's classification protocol**: this notebook uses3outerfolds and2innerfolds, not100stratified95/5splits, NCA or the authors' classifier. No significance claims or largest-score selection on test data.

## Inputs, models and averaging
- Broader cached data: **15EEG representations** from six regional100ms bins: full time course, early/middle/late windows, each scalp region, regional means/shape statistics, and per-trial amplitude-normalized bins. The original cache is region-major; the code verifies this geometry.
- Positive/negative also reuses the completed sweep's **20additional high-resolution inputs** (all-channel windows, narrow regions, coarse band energy and GFP) when available. No fresh EEG reading. Full-head224–304ms logistic remains an explicit **fixed reference**, displayed beside the automatically selected result.
- **Seven model families / nine settings:** logistic C=0.1,1,10; shrinkage LDA; approximate RBF-kernel classifier; Extra Trees; Random Forest; small32-unit neural network; distance-weighted kNN. The RBF approximation limits kernel cost on the larger pool. No raw waveform CNN in this cache-only notebook.
- **Reader identity:** individual, exact10, exact40, and **maximum40** averages. **Duration200:** same; sensitivity cutoffs150/250: individual/exact10. **Binary sentiment:** individual, exact5/10/20/40, maximum40. **Three-class and neutral/nonneutral:** individual/exact10/exact40/maximum40. Exact sizes discard remainders. Maximum40 retains smaller groups and reports actual sizes, so it must be compared separately.
- C1logistic screens all inputs inside training data. The best3inputs advance to9model settings; outer test folds evaluate the training-selected winner alongside fixed EEG, pre-onset EEG, metadata-only and dummy controls. All scaling/imputation/kernel maps fit training examples only. Multiclass models and balanced accuracy support all12reader classes.

## Storage, outputs and interpretation
**At most50MB new Drive outputs** in `Thesis/Results/word_sentiment_eeg/teco_other_targets/`. Reuse existing caches read-only; no new EEG copy or feature cache, no saved model bank, and no deletion. Compressed checkpoints retain memberships, predictions, configurations and metrics. Wrong checksums stop the run. Per-target sample/class counts, reader confusion matrix, all final scores, chosen models and unavailable cases are displayed below. Logs show reader X/12, target X/7, split/fold and setup progress.

Reader identification can exploit stable recording/session characteristics rather than person-specific brain activity. Duration classification can exploit eye movements and overlapping responses. These are diagnostics of information in this prepared EEG, not proof of a cognitive mechanism. The dataset has been explored already, so these are exploratory analyses.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/teco_other_targets'
REPO=Path('/content/word-sentiment-eeg-teco-other-targets')
CODE_REVISION='PIN_AFTER_TEST'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
# BOOTSTRAP_START: also exercised verbatim in cloud CI.
# Use a file-rendering backend in subprocesses, not Colab's unavailable inline plugin.
os.environ['MPLBACKEND']='Agg'
ENV=Path('/content/word-eeg-paper-classification-python')
PYTHON=ENV/'bin/python'
def setup_command(command):
    print('SETUP:', ' '.join(map(str,command)), flush=True)
    result=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Scientific environment setup failed; the full diagnostic is printed above. No EEG processing started.')
# Colab may lack ensurepip. This also repairs a partially created environment.
setup_command([sys.executable,'-m','venv','--without-pip',str(ENV)])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               '--only-binary=:all:','-r',str(REPO/'requirements-paper-classification.txt')])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'check'])
setup_command([str(PYTHON),'-c',
    'import numpy,pandas,scipy,sklearn,matplotlib; '
    'print("Scientific environment ready:",numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,matplotlib.__version__)'])
# BOOTSTRAP_END
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,50_000_000-used):
    raise RuntimeError('Reserve at most 50 MB for results. This notebook reads existing feature caches only; no EEG shards or extraction.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 50 MB.')
print('New output folder:',OUT)


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/teco_other_targets.py'),'--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='TeCo reader identity, fixation duration and familiar sentiment')


In [ ]:
import hashlib, gzip
from IPython.display import display, Markdown, Image
state=json.loads((OUT/'complete.json').read_text())
assert state['status']=='completed'
for name,sha in state['checksums'].items():
    if hashlib.sha256((OUT/name).read_bytes()).hexdigest()!=sha:raise RuntimeError('Checksum failed: '+name)
summary=pd.read_csv(OUT/'summary.csv',dtype={'size':str})
display(Markdown('## Finished and verified — all held-out test results'))
for target,g in summary.groupby('target',sort=False):
    display(Markdown('### '+target))
    with pd.option_context('display.max_rows',200,'display.max_columns',20):display(g.round(3))
display(Image(filename=str(OUT/'overview.png')))
if (OUT/'reader_confusion.png').is_file():display(Image(filename=str(OUT/'reader_confusion.png')))


In [ ]:
display(Markdown('## Target definitions and sample counts'))
for path in sorted((OUT/'targets').glob('*/definition.json')):
    value=json.loads(path.read_text());print(path.parent.name, json.dumps(value,ensure_ascii=False))
display(Markdown('## Chosen configurations — compare selected_eeg with fixed_eeg'))
rows=json.loads(gzip.decompress((OUT/'outer_results.json.gz').read_bytes()))
chosen=pd.DataFrame([{k:r.get(k) for k in ['target','mode','fold','size','variant','input','model','status','balanced_accuracy','test_examples','actual_group_min','actual_group_max','reason']} for r in rows])
with pd.option_context('display.max_rows',300):display(chosen[chosen.variant.isin(['selected_eeg','fixed_eeg'])].round(3))
display(Markdown('## Unavailable conditions'))
missing=pd.read_csv(OUT/'unavailable.csv')
with pd.option_context('display.max_rows',200):display(missing)
print('Saved results:', OUT)
print('summary.csv = final scores; development.csv.gz = validation search; outer_results.json.gz = predictions; fits/ = resumable compressed checkpoints and exact event memberships.')
